In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from pathlib import Path
import pandas as pd
from shapely.geometry import Polygon, Point
import scanpy as sc

In [ ]:
import os
data_dir = "/saving_directory_name"  # Change this to your actual data directory
os.makedirs(data_dir, exist_ok=True)
os.chdir(data_dir)  

In [ ]:
# ============================================================================
# 1. DATA LOADERS 
# ============================================================================
def load_cell_data(data_path: Path) -> pd.DataFrame:
    """Load Xenium cell data from parquet or CSV."""
    print("Loading Xenium cell data...")
    if (data_path / "cells.parquet").exists():
        cells_df = pd.read_parquet(data_path / "cells.parquet")
        print(f"Loaded cells.parquet: {len(cells_df)} cells")
    else:
        cells_df = pd.read_csv(data_path / "cells.csv.gz")
        print(f"Loaded cells.csv.gz: {len(cells_df)} cells")
    return cells_df


def load_tissue_selections(selections_path: Path) -> pd.DataFrame:
    """Load tissue selection coordinates from CSV."""
    print("\nLoading tissue selections...")
    selections_df = pd.read_csv(selections_path, skiprows=2)
    print(f"Columns: {selections_df.columns.tolist()}")
    print(f"Total coordinate rows: {len(selections_df)}")
    print(f"Unique selections: {selections_df['Selection'].unique()}")
    return selections_df

In [ ]:
# ============================================================================
# 2. CORE HELPERS
# ============================================================================

def get_centroid_cols(df: pd.DataFrame) -> list:
    """Finds and returns the existing coordinate column pair."""
    for cols in [('x_centroid', 'y_centroid'), ('centroid_x', 'centroid_y'), ('x', 'y'), ('X', 'Y')]:
        if cols[0] in df.columns and cols[1] in df.columns:
            return list(cols)
    raise ValueError(f"Centroid columns not found. Available: {df.columns.tolist()}")


def filter_cells(cells_df: pd.DataFrame, polygon: Polygon, coord_cols: list) -> pd.DataFrame:
    """Fast bounding-box pre-filter followed by precise polygon masking."""
    minx, miny, maxx, maxy = polygon.bounds
    # Drop ~90% of cells instantly via vectorized box limits
    bbox_mask = cells_df[coord_cols[0]].between(minx, maxx) & cells_df[coord_cols[1]].between(miny, maxy)
    subset = cells_df[bbox_mask]
    
    # Check exact shape boundaries on remaining subset
    poly_mask = [polygon.contains(Point(x, y)) for x, y in subset[coord_cols].values]
    return subset[poly_mask].copy()


def clean_name(name: str) -> str:
    """Sanitizes names for standard filesystem paths, replacing non-alphanumeric with underscores."""
    return re.sub(r'[^a-zA-Z0-9_-]', '_', name.strip())

In [ ]:
# ============================================================================
# 3. RUNTIME PIPELINE
# ============================================================================

def main():
    # --- CONFIGURATION & PATHS ---
    #Raw dataset path
    base_dir = Path("/gpfs/gibbs/pi/konnikova/Leo/maddi_xenium/fcb.ycga.yale.edu:3010/PBlED1s89zmi8w0pf6pve69x11ml8q5/20250801_Mss232_RQ35002_HsXP")
    slide2_dir = base_dir / "RQ35002-002_Slide-2/output-XETG00201__0042027__Region_1__20250801__181216"
    #CSV coordinate file generated from manual selection Xenium explorer
    selections_path = base_dir / "RQ35002-002_Slide-2/slide2_coordinates.csv"
    h5_path = slide2_dir / "cell_feature_matrix.h5"
    
    out_tissues = Path("slide2_individual_tissues")
    out_anndata = Path("slide2_tissue_anndata")
    out_tissues.mkdir(parents=True, exist_ok=True)
    out_anndata.mkdir(parents=True, exist_ok=True)

    # --- 1. LOAD SPATIAL DATA ---
    cells_df = load_cell_data(slide2_dir)
    selections_df = load_tissue_selections(selections_path)
    coord_cols = get_centroid_cols(cells_df)
    
    # Ensure cell_id is set as index cleanly
    cells_df_indexed = cells_df.set_index('cell_id') if 'cell_id' in cells_df.columns else cells_df.copy()
    
    # --- 2. PROCESS TISSUE POLYGONS ---
    cell_to_tissue = {}  # Map for quick alignment later: cell_id -> tissue_name
    summary_data = []
    unique_selections = selections_df['Selection'].unique()
    
    for sel_name in unique_selections:
        print(f"\nProcessing ROI: {sel_name}")
        try:
            coords = selections_df[selections_df['Selection'] == sel_name][['X', 'Y']].values
            tissue_cells = filter_cells(cells_df, Polygon(coords), coord_cols)
            
            if not tissue_cells.empty:
                tissue_cells['tissue_id'] = sel_name
                tissue_cells['selection_name'] = sel_name
                
                # Record cell relationships for AnnData step
                cell_ids = tissue_cells['cell_id'] if 'cell_id' in tissue_cells.columns else tissue_cells.index
                for cid in cell_ids:
                    cell_to_tissue[cid] = sel_name
                
                # Save Individual Tissue CSV
                safe_name = clean_name(sel_name)
                tissue_cells.to_csv(out_tissues / f"{safe_name}_cells.csv", index=False)
                summary_data.append({'tissue_name': sel_name, 'n_cells': len(tissue_cells), 'filename': f"{safe_name}_cells.csv"})
                print(f"  Selected and saved {len(tissue_cells)} cells.")
        except Exception as e:
            print(f"  Error processing {sel_name}: {e}")

    pd.DataFrame(summary_data).to_csv("slide2_tissue_summary.csv", index=False)

    # --- 3. LOAD & BUILD UNIFIED ANNDATA ---
    print("\nLoading 10x cell feature matrix...")
    adata_full = sc.read_10x_h5(h5_path)
    adata_full.var_names_unique = True

    # Align overlapping metrics seamlessly
    common_cells = cells_df_indexed.index.intersection(adata_full.obs.index)
    if common_cells.empty:
        print("WARNING: No matching cell IDs found between matrix and dataframe indices.")
        return

    # Slice and load shared spatial properties globally 
    adata_full = adata_full[common_cells, :].copy()
    adata_full.obs = pd.concat([adata_full.obs, cells_df_indexed.loc[common_cells]], axis=1)
    adata_full.obsm['spatial'] = adata_full.obs[coord_cols].values
    
    # Map spatial ROIs onto your expression observation indices
    adata_full.obs['tissue_id'] = adata_full.obs.index.map(cell_to_tissue)
    adata_full.obs['selection_name'] = adata_full.obs['tissue_id']

    # --- 4. EXPORT SUBSETTED ANNDATA OBJECTS & DIAGNOSTICS ---
    print("\nSplitting and exporting tissue AnnData objects...")
    total_cells = len(cell_to_tissue)
    total_umis = 0 

    print(f"\n=== FINAL SUMMARY ===")
    for t_name in unique_selections:
        # Clean subset via Pandas mapping column instead of manual intersection loops
        adata_tissue = adata_full[adata_full.obs['tissue_id'] == t_name].copy()
        
        if adata_tissue.n_obs == 0:
            print(f"  WARNING: No cells with expression data found for {t_name}")
            continue

        # Save H5AD matrix
        safe_name = clean_name(t_name).lower()
        adata_tissue.write_h5ad(out_anndata / f"{safe_name}.h5ad")

        # Terminal Statistics Computations (Handles sparse vs dense arrays smoothly)
        import scipy.sparse as sp
        tissue_umis = adata_tissue.X.sum() if not sp.issparse(adata_tissue.X) else adata_tissue.X.sum()
        total_umis += tissue_umis
        
        pct_cells = (adata_tissue.n_obs / total_cells) * 100 if total_cells else 0
        mean_umis = tissue_umis / adata_tissue.n_obs

        print(f"  {t_name}:")
        print(f"    Cells: {adata_tissue.n_obs:,} ({pct_cells:.1f}%) | Genes: {adata_tissue.n_vars:,} | Total UMIs: {tissue_umis:,.0f} | Avg UMI/Cell: {mean_umis:.0f}")
        print(f"    Spatial Limits: X({adata_tissue.obs[coord_cols[0]].min():.0f}-{adata_tissue.obs[coord_cols[0]].max():.0f}), Y({adata_tissue.obs[coord_cols[1]].min():.0f}-{adata_tissue.obs[coord_cols[1]].max():.0f})")

    print(f"\nOverall Processed Aggregates:")
    print(f"  Total cells across all tissues: {total_cells:,}")
    print(f"  Total UMIs across all tissues: {total_umis:,.0f}")
    if total_cells > 0:
        print(f"  Average UMIs per cell: {total_umis / total_cells:.0f}")


if __name__ == "__main__":
    main()